In [1]:
# @title pip installs
!pip install google-adk -q
!pip install google-cloud-aiplatform -q


In [2]:
# @title Constants

PROJECT_ID = "qwiklabs-gcp-04-50597814ce11"
LOCATION = "us-east4"
STAGING_BUCKET = "qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging"
DEFAULT_AGENT_MODEL = "gemini-2.5-flash"
USER_ID = "agent-engine-test-user"
MESSAGE = "Give me the news highlights in the world of .NET and C#."

In [3]:
# @title Vertex AI & GCS Setup
import vertexai
from google.cloud import storage

# Ensure the bucket exists
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(STAGING_BUCKET)
    print(f"Bucket {STAGING_BUCKET} already exists.")
except Exception:
    print(f"Creating bucket {STAGING_BUCKET}...")
    storage_client.create_bucket(STAGING_BUCKET, location=LOCATION)

vertexai.init(
   project=PROJECT_ID,
   location=LOCATION,
   staging_bucket=f"gs://{STAGING_BUCKET}",
)

Bucket qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging already exists.


In [4]:
# @title Agent
from google.adk.agents import Agent
from google.adk.tools import google_search
#from pydantic import ConfigDict

code_agent = Agent(
    name="CodeMonkey",
    model=DEFAULT_AGENT_MODEL,
    description=
      "You answer programming language questions.",
    instruction=
      """You answer questions related to programming.
       Answer coding questions using C# unless told otherwise.
       """,
    tools=[google_search]
)


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [5]:
# @title Test Agent Locally
from vertexai.preview.reasoning_engines import AdkApp
import textwrap

app = AdkApp(agent=code_agent)

response = []

for event in app.stream_query(
    user_id=USER_ID,
    message=MESSAGE,
):
    # Safely navigate the nested dictionary structure
    content = event.get("content", {})
    parts = content.get("parts", [])

    for part in parts:
        # Check if 'text' exists in the part dictionary
        if isinstance(part, dict) and "text" in part:
            response.append(part["text"])
        elif isinstance(part, str):
            response.append(part)

# Wrap text at 80 columns before printing
wrapper = textwrap.TextWrapper(width=80)
for item in response:
    wrapped_item = wrapper.fill(text=item)
    print(wrapped_item)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:872: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


Here's a roundup of the latest news and highlights in the world of .NET and C#:
**C# Language Evolution (C# 13, 14, and 15)**  The C# language continues to
evolve rapidly, with several versions seeing active development and preview
releases.  *   **C# 13** was announced with preview features at Microsoft Build
2024, focusing on enhanced `params`, new extension types, and performance and
memory improvements for .NET developers. *   **C# 14**, shipping with .NET 10,
introduces significant new features aimed at cleaner, safer, and more productive
code. Key updates include extension members (allowing extension properties and
static extension members), null-conditional assignment, improved implicit
conversions for `Span<T>` and `ReadOnlySpan<T>`, and enhanced compiler analysis
for nullable references to reduce `NullReferenceException`s. This version also
supports `nameof` for unbound generic types, modifiers on simple lambda
parameters, field-backed properties, partial events and constructo

In [6]:
# @title Deploy to Agent Platform

from vertexai import agent_engines

# The staging_bucket is handled by the global vertexai.init configuration
remote_agent = agent_engines.create(
   app,
   requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)


INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.157.0', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creati

In [7]:
# @title Test Remote Agent

print(f"Targeting remote agent: {remote_agent.resource_name}")

# List available methods to verify the correct one
methods = [method for method in dir(remote_agent) if not method.startswith('_')]
print(f"Available methods: {methods}")

try:
    print("\n--- Attempting Query ---")
    # In some SDK versions, the method is 'query' or 'predict'.
    # For ADK remote engines, 'stream_query' is the standard.
    response_iterator = remote_agent.stream_query(
       user_id=USER_ID,
       message=MESSAGE,
    )

    for i, event in enumerate(response_iterator):
        # Extract text safely from the event
        content = event.get("content", {})
        parts = content.get("parts", [])
        for part in parts:
            if isinstance(part, dict) and "text" in part:
                print(part["text"], end="")
            elif isinstance(part, str):
                print(part, end="")

except Exception as e:
    print(f"\nError: {e}")

Targeting remote agent: projects/172082414361/locations/us-east4/reasoningEngines/2790762821434998784
Available methods: ['api_client', 'async_add_session_to_memory', 'async_create_session', 'async_delete_session', 'async_get_session', 'async_list_sessions', 'async_search_memory', 'async_stream_query', 'client_class', 'create', 'create_session', 'create_time', 'credentials', 'delete', 'delete_session', 'display_name', 'encryption_spec', 'execution_api_client', 'execution_async_client', 'gca_resource', 'get_session', 'labels', 'list', 'list_sessions', 'location', 'name', 'operation_schemas', 'project', 'resource_name', 'stream_query', 'streaming_agent_run_with_events', 'to_dict', 'update', 'update_time', 'wait']

--- Attempting Query ---
